# Section 05d Onboarding Notebook: Analytical Nuclear Gradients — Pulay Forces, Energy-Weighted Density Matrices, CPHF/Z-Vector MP2 Gradients & Coupled Cluster $\Lambda$-Equations

### Overview & Pedagogical Design
Computing nuclear forces $\mathbf{F}_A = -\nabla_{\mathbf{R}_A} E$ via numerical finite differences requires $6N_{\text{atoms}}$ separate single-point calculations, which becomes prohibitively expensive for geometry optimizations, transition-state searches, and *ab initio* molecular dynamics. **Analytical nuclear gradients** compute exact forces across all $3N_{\text{atoms}}$ coordinates in a fraction of the cost of a single energy evaluation ([J. Chem. Theory Comput. 2016](https://pubs.acs.org/doi/10.1021/acs.jctc.6b00015); [J. Chem. Phys. 128, 154101](https://pubs.aip.org/aip/jcp/article/128/15/154101/70588/An-atomic-orbital-based-reformulation-of-energy)):

1. **Variational Wavefunctions (RHF / Kohn–Sham DFT) & Pulay's Theorem**:
   Because the RHF energy is stationary with respect to orbital rotations ($\frac{\partial E_{\text{RHF}}}{\partial \mathbf{C}} = \mathbf{0}$ subject to $\mathbf{C}^\mathsf{T}\mathbf{S}\mathbf{C} = \mathbf{I}$), we **never** need to differentiate the MO coefficients $\frac{\partial \mathbf{C}}{\partial R_{A\xi}}$! Instead, the orthonormality constraint $\mathbf{C}^\mathsf{T}\frac{\partial \mathbf{S}}{\partial R_{A\xi}}\mathbf{C} + \frac{\partial \mathbf{C}^\mathsf{T}}{\partial R_{A\xi}}\mathbf{S}\mathbf{C} + \mathbf{C}^\mathsf{T}\mathbf{S}\frac{\partial \mathbf{C}}{\partial R_{A\xi}} = \mathbf{0}$ replaces all MO-coefficient derivatives with the **Energy-Weighted Density Matrix** $\mathbf{W}^{\text{RHF}} = 2\sum_{i=1}^{n_{\text{occ}}} \varepsilon_i \mathbf{c}_i \mathbf{c}_i^\mathsf{T}$ contracted against the overlap derivative $-\text{Tr}\!\left(\mathbf{W}^{\text{RHF}}\frac{\partial \mathbf{S}}{\partial R_{A\xi}}\right)$ (**Pulay force**).
2. **Non-Variational Wavefunctions (MP2) & the Handy–Schaefer Z-Vector Method**:
   Because $E_{\text{MP2}}$ is *not* stationary with respect to $\mathbf{C}$, its gradient depends on the orbital response $\frac{\partial \mathbf{C}}{\partial R_{A\xi}}$ governed by the **Coupled-Perturbed Hartree–Fock (CPHF)** equations $\mathbf{A}\,\mathbf{U}^{[x]} = \mathbf{B}^{[x]}$. Handy and Schaefer's **Z-vector method** eliminates all $3N_{\text{atoms}}$ CPHF equations in favor of a **single** perturbation-independent linear system $\mathbf{A}^\mathsf{T}\mathbf{Z} = -\mathbf{L}^{\text{MP2}}$ to form the **relaxed MP2 1-RDM** $\mathbf{P}^{\text{MP2,rel}}$!
3. **Coupled Cluster Gradients & the $\Lambda$-Equations**:
   Because Coupled Cluster ($e^{-\hat{T}}\hat{H}e^{\hat{T}}$) is non-variational with respect to the cluster amplitudes $t_\mu$, we construct the variational **CC Lagrangian** $\mathcal{L}_{\text{CC}}(T, \Lambda) = \langle\Phi_0|(1 + \hat{\Lambda})e^{-\hat{T}}\hat{H}e^{\hat{T}}|\Phi_0\rangle$ and solve the linear left-eigenstate $\Lambda$-equations $\frac{\partial \mathcal{L}_{\text{CC}}}{\partial t_\mu} = 0$.

In this notebook, **no gradient or response functions are pre-solved for you**. You will implement **15 granular `# TODO` functions**:
- **Part 1 — Nuclear Repulsion Gradients & AO Basis Derivative Integrals (`TODO 1` – `TODO 3`)**
- **Part 2 — Analytical RHF Gradients, Energy-Weighted Density Matrix $\mathbf{W}^{\text{RHF}}$ & Hellmann–Feynman vs. Pulay Decomposition (`TODO 4` – `TODO 7`)**
- **Part 3 — CPHF Electronic Hessian, Handy–Schaefer Z-Vector Equation & Analytical MP2 Gradients (`TODO 8` – `TODO 12`)**
- **Part 4 — Coupled Cluster $\Lambda$-Equations & Quasi-Newton BFGS Geometry Optimization (`TODO 13` – `TODO 15`)**
- **Part 5 — Step-by-Step Checkpoint & Verification Suite (`run_section05d_checkpoints`)**

In [ ]:
import numpy as np
from scipy.linalg import eigh
from scipy.special import erf
from pyscf import gto, scf, mp, fci, grad
import matplotlib.pyplot as plt

np.set_printoptions(precision=6, suppress=True, linewidth=110)
print("Section 05d Analytical Gradients environment initialized with PySCF. Implement TODOs 1-15 and run `run_section05d_checkpoints()`!")

## Part 1: Nuclear Repulsion Gradients & AO Integral Derivatives (TODOs 1–3)

### Mathematical Specifications
1. **Nuclear Repulsion Energy & Analytical Gradient (`compute_nuclear_repulsion_and_grad` — TODO 1)**:
   $$E_{\text{nuc}} = \sum_{A < B} \frac{Z_A Z_B}{R_{AB}}, \qquad \frac{\partial E_{\text{nuc}}}{\partial \mathbf{R}_A} = -\sum_{B \ne A} Z_A Z_B \frac{\mathbf{R}_A - \mathbf{R}_B}{\|\mathbf{R}_A - \mathbf{R}_B\|_2^3}$$
2. **Exact 1D Gaussian Center-Derivative Rule (`gaussian_1d_overlap_and_center_deriv` — TODO 2)**:
   For a 1D primitive Gaussian $g_i(x; \alpha, A_x) = (x - A_x)^i e^{-\alpha(x - A_x)^2}$, differentiating with respect to the nuclear center $A_x$ raises and lowers angular momentum:
   $$\frac{\partial g_i(x; \alpha, A_x)}{\partial A_x} = 2\alpha\,g_{i+1}(x; \alpha, A_x) - i\,g_{i-1}(x; \alpha, A_x)$$
   Therefore, the derivative of the 1D overlap integral $S_{ij}^x = \langle g_i(\alpha, A_x) | g_j(\beta, B_x) \rangle$ with respect to $A_x$ is:
   $$\frac{\partial S_{ij}^x}{\partial A_x} = 2\alpha\,S_{i+1, j}^x - i\,S_{i-1, j}^x, \qquad \frac{\partial S_{ij}^x}{\partial B_x} = -\frac{\partial S_{ij}^x}{\partial A_x}$$
3. **Diatomic STO-3G Integral & Derivative Generator (`compute_diatomic_sto3g_integrals_and_derivs` — TODO 3)**:
   Given internuclear distance $R$ (in Bohr) for $\text{HeH}^+$ ($\zeta_{\text{He}} = 2.0925, \zeta_{\text{H}} = 1.24$, charges $Z_{\text{He}} = 2, Z_{\text{H}} = 1$), evaluate $(\mathbf{S}, \mathbf{T}, \mathbf{V}, \mathbf{H}^{\text{core}}, \text{eri}, E_{\text{nuc}})$ and their derivatives with respect to $R$: $(\frac{d\mathbf{S}}{dR}, \frac{d\mathbf{T}}{dR}, \frac{d\mathbf{V}}{dR}, \frac{d\mathbf{H}^{\text{core}}}{dR}, \frac{d\text{eri}}{dR}, \frac{dE_{\text{nuc}}}{dR})$.

In [ ]:
def compute_nuclear_repulsion_and_grad(
    atom_charges: np.ndarray, atom_coords_bohr: np.ndarray
) -> tuple[float, np.ndarray]:
    '''
    TODO 1: Compute the nuclear repulsion energy E_nuc (float) and analytical nuclear gradient
    dE_nuc / dR of shape (N_atoms, 3) in Hartree / Bohr. Return (e_nuc, grad_e_nuc).
    '''
    # TODO 1: Loop over atom pairs (A, B) with A != B, accumulate E_nuc and -Z_A * Z_B * (R_A - R_B) / ||R_A - R_B||^3.
    raise NotImplementedError("Complete TODO 1: compute_nuclear_repulsion_and_grad")


def gaussian_1d_overlap_and_center_deriv(
    i: int, j: int, A_x: float, B_x: float, alpha: float, beta: float
) -> tuple[float, float, float]:
    '''
    TODO 2: Compute the 1D unnormalized primitive Gaussian overlap S_{i,j}^x and its analytical
    derivatives dS_{i,j}^x / dA_x and dS_{i,j}^x / dB_x for angular momenta i, j in {0, 1}.
    Return (S_ij, dS_dAx, dS_dBx).
    '''
    # TODO 2: Implement 1D Hermite overlap S(i, j) for i, j <= 2, then evaluate
    #         dS_dAx = 2 * alpha * S(i + 1, j) - (i * S(i - 1, j) if i > 0 else 0.0) and dS_dBx = -dS_dAx.
    raise NotImplementedError("Complete TODO 2: gaussian_1d_overlap_and_center_deriv")


def _heh_pyscf_integrals_at_r(R_bohr: float) -> dict[str, np.ndarray | float | object]:
    '''Build a PySCF Mole object for HeH+ (STO-3G) at bond length R_bohr and extract AO integrals.'''
    mol = gto.M(
        atom=f"He 0.0 0.0 0.0; H {R_bohr} 0.0 0.0",
        unit="Bohr",
        basis="sto-3g",
        charge=1,
        spin=0,
        verbose=0,
    )
    S = mol.intor("int1e_ovlp")
    T = mol.intor("int1e_kin")
    V = mol.intor("int1e_nuc")
    eri = mol.intor("int2e")
    return {"mol": mol, "S": S, "T": T, "V": V, "H_core": T + V, "eri": eri, "e_nuc": float(mol.energy_nuc())}


def compute_diatomic_sto3g_integrals_and_derivs(
    R_bohr: float = 1.4632, eps_bohr: float = 1e-5
) -> dict[str, np.ndarray | float | object]:
    '''
    TODO 3: Using `_heh_pyscf_integrals_at_r(R_bohr)`, return a dictionary containing the PySCF
    integrals at `R_bohr` ('mol', 'S', 'T', 'V', 'H_core', 'eri', 'e_nuc') AND their bond-length
    derivatives ('dS_dR', 'dT_dR', 'dV_dR', 'dH_core_dR', 'deri_dR', 'de_nuc_dR') at R_bohr.
    '''
    # TODO 3: Evaluate _heh_pyscf_integrals_at_r at R_bohr, R_bohr + eps_bohr, and R_bohr - eps_bohr and form central derivatives.
    raise NotImplementedError("Complete TODO 3: compute_diatomic_sto3g_integrals_and_derivs")

## Part 2: Analytical Restricted Hartree–Fock (RHF) Gradients & Pulay Forces (TODOs 4–7)

### Mathematical Specifications
Differentiating $E_{\text{RHF}} = 2\text{Tr}(\mathbf{D}\mathbf{H}^{\text{core}}) + \text{Tr}[\mathbf{D}(2\mathbf{J}(\mathbf{D}) - \mathbf{K}(\mathbf{D}))] + E_{\text{nuc}}$ with respect to nuclear coordinate $R$ gives two contributions:
1. **Explicit Integral Derivatives** (differentiating $\mathbf{H}^{\text{core}}$ and $(\mu\nu|\lambda\sigma)$ at fixed $\mathbf{D}$):
   $$\left(\frac{\partial E_{\text{RHF}}}{\partial R}\right)_{\mathbf{D}} = 2\sum_{\mu\nu} D_{\mu\nu}\frac{\partial H_{\mu\nu}^{\text{core}}}{\partial R} + \sum_{\mu\nu\lambda\sigma} D_{\mu\nu}D_{\lambda\sigma}\left[2\frac{\partial(\mu\nu|\lambda\sigma)}{\partial R} - \frac{\partial(\mu\lambda|\nu\sigma)}{\partial R}\right] + \frac{\partial E_{\text{nuc}}}{\partial R}$$
2. **Implicit Density Response $\frac{\partial \mathbf{D}}{\partial R}$ $\longrightarrow$ Pulay Overlap Term**:
   Using the Roothaan–Hall equation $\mathbf{F}\mathbf{C}_{\text{occ}} = \mathbf{S}\mathbf{C}_{\text{occ}}\boldsymbol{\varepsilon}_{\text{occ}}$ and differentiating the MO orthonormality constraint $\mathbf{C}_{\text{occ}}^\mathsf{T}\mathbf{S}\mathbf{C}_{\text{occ}} = \mathbf{I}_{n_{\text{occ}}}$ yields:
   $$2\text{Tr}\!\left(\frac{\partial \mathbf{D}}{\partial R}\mathbf{F}\right) = -\text{Tr}\!\left(\mathbf{W}^{\text{RHF}}\frac{\partial \mathbf{S}}{\partial R}\right), \qquad \text{where} \quad W_{\mu\nu}^{\text{RHF}} = 2\sum_{i=0}^{n_{\text{occ}}-1} \varepsilon_i\,C_{\mu i}\,C_{\nu i}$$
   This $-\text{Tr}(\mathbf{W}^{\text{RHF}}\frac{\partial \mathbf{S}}{\partial R})$ term (combined with the AO center derivatives inside $\frac{\partial \mathbf{H}^{\text{core}}}{\partial R}$ and $\frac{\partial(\mu\nu|\lambda\sigma)}{\partial R}$) is the famous **Pulay force**!

In [ ]:
def compute_rhf_energy_weighted_density_matrix(
    C: np.ndarray, eps: np.ndarray, n_occ: int
) -> np.ndarray:
    '''
    TODO 4: Construct the spin-summed RHF energy-weighted density matrix
    W_{uv}^{RHF} = 2 * sum_{i=0}^{n_occ-1} eps_i * C_{u,i} * C_{v,i}.
    '''
    # TODO 4: Slice C_occ = C[:, :n_occ] and eps_occ = eps[:n_occ], and return 2.0 * (C_occ * eps_occ) @ C_occ.T.
    raise NotImplementedError("Complete TODO 4: compute_rhf_energy_weighted_density_matrix")


def assemble_analytical_rhf_gradient_scalar(
    D: np.ndarray,
    W_rhf: np.ndarray,
    dH_core_dR: np.ndarray,
    dS_dR: np.ndarray,
    deri_dR: np.ndarray,
    de_nuc_dR: float,
) -> dict[str, float]:
    '''
    TODO 5: Assemble the analytical RHF energy gradient dE_RHF / dR and return a dict with keys:
    'grad_1e', 'grad_2e', 'grad_pulay_ovlp', 'grad_nuc', and 'grad_total'.
    Specifically:
    - grad_1e         = 2 * Tr(D @ dH_core_dR)
    - grad_2e         = sum_{uvls} D_{uv} D_{ls} (2 * deri_dR[u,v,l,s] - deri_dR[u,l,v,s])
    - grad_pulay_ovlp = -Tr(W_rhf @ dS_dR)
    - grad_nuc        = de_nuc_dR
    - grad_total      = grad_1e + grad_2e + grad_pulay_ovlp + grad_nuc
    '''
    # TODO 5: Evaluate each of the 4 terms and their sum grad_total.
    raise NotImplementedError("Complete TODO 5: assemble_analytical_rhf_gradient_scalar")


def compare_hellmann_feynman_vs_full_analytical_grad(
    grad_dict: dict[str, float]
) -> dict[str, float]:
    '''
    TODO 6: Compare the full analytical RHF gradient (`grad_total`) against the approximation
    that omits the overlap Pulay term (`grad_without_ovlp_pulay = grad_1e + grad_2e + grad_nuc`).
    Return a dict with keys 'grad_total', 'grad_without_ovlp_pulay', and 'pulay_ovlp_magnitude'.
    '''
    # TODO 6: Extract the components from grad_dict and return the comparison dictionary.
    raise NotImplementedError("Complete TODO 6: compare_hellmann_feynman_vs_full_analytical_grad")


def verify_gradient_5pt_stencil(
    energy_of_r_fn, R_bohr: float, analytical_grad: float, h_step: float = 1e-3
) -> tuple[float, float]:
    '''
    TODO 7: Compute the 4th-order 5-point finite-difference derivative:
    dE/dR_5pt = (-E(R + 2h) + 8*E(R + h) - 8*E(R - h) + E(R - 2h)) / (12 * h)
    and return (dE_dR_5pt, abs(analytical_grad - dE_dR_5pt)).
    '''
    # TODO 7: Evaluate energy_of_r_fn at R +/- h and R +/- 2h and compute the 5-point stencil derivative.
    raise NotImplementedError("Complete TODO 7: verify_gradient_5pt_stencil")

## Part 3: Coupled-Perturbed Hartree–Fock (CPHF), Handy–Schaefer Z-Vector & Analytical MP2 Gradients (TODOs 8–12)

### Mathematical Specifications
For closed-shell MP2 ($E_{\text{corr}}^{\text{MP2}} = \sum_{ijab} t_{iajb}\big[2(ia|jb) - (ib|ja)\big]$, where $\tilde{\tau}_{iajb} = 2t_{iajb} - t_{ibja}$), differentiating with respect to nuclear coordinate $R$ requires ([J. Chem. Theory Comput. 2016](https://pubs.acs.org/doi/10.1021/acs.jctc.6b00015); [J. Chem. Phys. 128, 154101](https://pubs.aip.org/aip/jcp/article/128/15/154101/70588/An-atomic-orbital-based-reformulation-of-energy)):
1. **Unrelaxed MP2 Density Blocks & Non-Separable 2-RDM (`compute_mp2_unrelaxed_densities` — TODO 8)**:
   $$P_{ik}^{(2)} = -2\sum_{jab} t_{iajb}\,\tilde{\tau}_{kajb}, \qquad P_{ab}^{(2)} = +2\sum_{ijc} t_{iajc}\,\tilde{\tau}_{ibjc}, \qquad \Gamma_{iajb}^{(2)} = 2\,\tilde{\tau}_{iajb}$$
2. **CPHF Electronic Orbital Hessian $A_{ia,jb}$ & MP2 Orbital Lagrangian $L_{ai}$ (`build_cphf_hessian_and_mp2_lagrangian` — TODO 9)**:
   For occupied indices $i,j$ and virtual indices $a,b$, the singlet RHF orbital rotation Hessian is:
   $$A_{ia,jb} = (\varepsilon_a - \varepsilon_i)\delta_{ij}\delta_{ab} + 4(ia|jb) - (ij|ab) - (ib|ja)$$
   and the MP2 orbital-rotation Lagrangian driving term is:
   $$L_{ai} = 2\sum_{jbc} \tilde{\tau}_{ijbc}\,(ab|jc) - 2\sum_{jk b} \tilde{\tau}_{jkab}\,(ik|jb) + \sum_{kl} P_{kl}^{(2)}\big[2(ai|kl) - (ak|il)\big] + \sum_{bc} P_{bc}^{(2)}\big[2(ai|bc) - (ab|ic)\big]$$
3. **Handy–Schaefer Z-Vector Equation (`solve_z_vector_and_relaxed_mp2_1rdm` — TODO 10)**:
   Solve $\sum_{jb} A_{ia,jb}\,Z_{jb} = -L_{ai}$ for $Z_{ia}$ and assemble the full spin-summed **relaxed MP2 response 1-RDM** $\mathbf{P}^{(2),\text{rel}}$ in the MO basis (with $P_{ij}^{(2)}$ in `[:n_occ, :n_occ]`, $P_{ab}^{(2)}$ in `[n_occ:, n_occ:]`, and $Z_{ia}$ in `[:n_occ, n_occ:]` and `[n_occ:, :n_occ]`).
4. **MP2 Energy-Weighted Density Matrix $\mathbf{W}^{(2)}$ & Analytical MP2 Gradient (`build_mp2_ewdm_and_gradient` — TODOs 11–12)**:
    Using the exact Lagrangian stationarity of the relaxed MP2 functional, evaluate the analytical MP2 nuclear gradient and verify it against the 5-point finite-difference derivative of $E_{\text{MP2}}(R)$!

In [ ]:
def compute_mp2_unrelaxed_densities(
    eri_mo: np.ndarray, eps: np.ndarray, n_occ: int
) -> dict[str, np.ndarray | float]:
    '''
    TODO 8: Compute MP2 amplitudes t2 (shape n_occ, n_virt, n_occ, n_virt), tau_tilde = 2*t2 - swapaxes(t2,1,3),
    occupied-occupied density block P_oo^{(2)}, virtual-virtual density block P_vv^{(2)},
    and two-body cumulant Gamma_{iajb}^{(2)} = 2 * tau_tilde.
    Return a dict with keys 'e_mp2_corr', 't2', 'tau_tilde', 'P_oo', 'P_vv', 'Gamma_ovov'.
    '''
    # TODO 8: Slice iajb = eri_mo[:n_occ, n_occ:, :n_occ, n_occ:], form t2, tau_tilde, P_oo, P_vv, and Gamma_ovov.
    raise NotImplementedError("Complete TODO 8: compute_mp2_unrelaxed_densities")


def build_cphf_hessian_and_mp2_lagrangian(
    eri_mo: np.ndarray,
    eps: np.ndarray,
    t2: np.ndarray,
    tau_tilde: np.ndarray,
    P_oo: np.ndarray,
    P_vv: np.ndarray,
    n_occ: int,
) -> tuple[np.ndarray, np.ndarray]:
    '''
    TODO 9: Construct the CPHF orbital Hessian A_cphf of shape (n_occ*n_virt, n_occ*n_virt)
    and the MP2 orbital Lagrangian rhs vector L_ai of shape (n_occ, n_virt).
    Return (A_cphf, L_ai).
    '''
    # TODO 9: Build the 4-index A_{i,a,j,b} = (eps_a - eps_i)*delta_ij*delta_ab + 4*(ia|jb) - (ij|ab) - (ib|ja),
    #         reshape to (n_occ*n_virt, n_occ*n_virt), and evaluate L_ai (shape n_occ, n_virt).
    raise NotImplementedError("Complete TODO 9: build_cphf_hessian_and_mp2_lagrangian")


def solve_z_vector_and_relaxed_mp2_1rdm(
    A_cphf: np.ndarray, L_ai: np.ndarray, P_oo: np.ndarray, P_vv: np.ndarray
) -> tuple[np.ndarray, np.ndarray]:
    '''
    TODO 10: Solve A_cphf @ Z_flat = -L_ai.ravel() for Z_{ia} (shape n_occ, n_virt),
    and assemble the full symmetric (n_mo, n_mo) relaxed MP2 correlation 1-RDM P_mp2_corr_mo
    with P_oo in [:n_occ, :n_occ], P_vv in [n_occ:, n_occ:], Z in [:n_occ, n_occ:], and Z.T in [n_occ:, :n_occ].
    Return (Z_ia, P_mp2_corr_mo).
    '''
    # TODO 10: Solve the linear system via np.linalg.solve, reshape Z_ia, and populate P_mp2_corr_mo.
    raise NotImplementedError("Complete TODO 10: solve_z_vector_and_relaxed_mp2_1rdm")


def compute_mp2_energy_weighted_density_mo(
    eri_mo: np.ndarray,
    eps: np.ndarray,
    tau_tilde: np.ndarray,
    P_mp2_corr_mo: np.ndarray,
    n_occ: int,
) -> np.ndarray:
    '''
    TODO 11: Construct the symmetric (n_mo, n_mo) MP2 correlation contribution W_mp2_corr_mo
    to the energy-weighted density matrix in the MO basis:
    - W_{ij} = -2 * sum_{kab} tau_tilde_{iakb} * (ja|kb) - 0.5 * (eps_i + eps_j) * P_{ij}^{(2)}
               - sum_{pq} P_{pq}^{(2),rel} * [2*(ij|pq) - (ip|jq)]
    - W_{ab} = -2 * sum_{ijc} tau_tilde_{iajc} * (ib|jc) - 0.5 * (eps_a + eps_b) * P_{ab}^{(2)}
    - W_{ia} = W_{ai} = -2 * sum_{jkb} tau_tilde_{jkab} * (ij|kb) - eps_i * Z_{ia}
    Return W_mp2_corr_mo.
    '''
    # TODO 11: Populate the (occ,occ), (virt,virt), and (occ,virt)/(virt,occ) blocks of W_mp2_corr_mo.
    raise NotImplementedError("Complete TODO 11: compute_mp2_energy_weighted_density_mo")


def assemble_analytical_mp2_gradient_scalar(
    R_bohr: float,
) -> dict[str, float]:
    '''
    TODO 12: Evaluate the complete analytical MP2 gradient dE_MP2 / dR = dE_RHF / dR + dE_corr^{MP2} / dR
    for HeH+ (STO-3G) at bond length `R_bohr` using the Hellmann-Feynman-Pulay / CPHF response relation.
    Return a dict with keys 'e_rhf', 'e_mp2_tot', 'grad_rhf', 'grad_mp2_corr', and 'grad_mp2_total'.
    '''
    # TODO 12: Run RHF at R_bohr, compute analytical grad_rhf (TODO 5), evaluate the orbital + integral
    #          derivative response of E_MP2, and return the gradient dictionary.
    raise NotImplementedError("Complete TODO 12: assemble_analytical_mp2_gradient_scalar")

## Part 4: Coupled Cluster Lagrangian ($\Lambda$-Equations) & BFGS Geometry Optimization (TODOs 13–15)

### Mathematical Specifications
1. **Coupled Cluster $\Lambda$-Equations & Relaxed 1-RDM (`solve_ccd_lambda_and_1rdm_2x2` — TODO 13)**:
   For a 2-orbital, 2-electron closed-shell model ($\text{HeH}^+$ `STO-3G`), the CCD/CCSD doubles amplitude $t = t_{00}^{11}$ satisfies the quadratic Riccati equation:
   $$R(t) = (01|01) + 2(\varepsilon_1 - \varepsilon_0 + \dots)\,t + (01|01)\,t^2 = 0$$
   Differentiating the CC Lagrangian $\mathcal{L}_{\text{CC}}(t, \lambda) = E_{\text{corr}}(t) + \lambda\,R(t)$ with respect to $t$ gives the linear **$\Lambda$-equation**:
   $$\frac{\partial \mathcal{L}_{\text{CC}}}{\partial t} = \frac{\partial E_{\text{corr}}}{\partial t} + \lambda \frac{\partial R}{\partial t} = 0 \quad \Longrightarrow \quad \lambda = -\frac{\partial E_{\text{corr}}/\partial t}{\partial R/\partial t}$$
   from which the bi-variational CC 1-RDM occupation numbers are $P_{00}^{\text{CC}} = 2(1 - \lambda t)$ and $P_{11}^{\text{CC}} = 2\lambda t$!
2. **BFGS Inverse-Hessian Update (`bfgs_inverse_hessian_update` — TODO 14)** & **Geometry Optimization Driver (`run_geometry_optimization_1d` — TODO 15)**:
   Given step $\mathbf{s}_k = \mathbf{x}_{k+1} - \mathbf{x}_k$ and gradient change $\mathbf{y}_k = \mathbf{g}_{k+1} - \mathbf{g}_k$ (with $\rho_k = 1 / (\mathbf{y}_k^\mathsf{T}\mathbf{s}_k) > 0$), update the inverse Hessian approximation $\mathbf{H}_k^{-1}$ via the BFGS formula:
   $$\mathbf{H}_{k+1}^{-1} = \left(\mathbf{I} - \rho_k \mathbf{s}_k \mathbf{y}_k^\mathsf{T}\right)\mathbf{H}_k^{-1}\left(\mathbf{I} - \rho_k \mathbf{y}_k \mathbf{s}_k^\mathsf{T}\right) + \rho_k \mathbf{s}_k \mathbf{s}_k^\mathsf{T}$$
   and find the equilibrium bond length $R_{\text{eq}}$ where $|dE/dR| < 10^{-6}\ E_{\text{h}}/\text{Bohr}$!

In [ ]:
def solve_ccd_lambda_and_1rdm_2x2(
    h_mo: np.ndarray, eri_mo: np.ndarray
) -> dict[str, float | np.ndarray]:
    '''
    TODO 13: For a 2-electron, 2-spatial-orbital system (HeH+ STO-3G), solve the exact coupled-cluster
    doubles Riccati equation R(t) = K_01 + 2 * Delta_01 * t + K_01 * t^2 = 0 for the ground-state
    amplitude `t`, then solve the linear CC Lambda equation dL/dt = 0 for `lam`, and construct
    the bi-variational CC 1-RDM P_cc_mo = diag(2*(1 - lam*t), 2*lam*t).
    Here H_00 = 2*h_00 + (00|00), H_11 = 2*h_11 + (11|11), Delta_01 = 0.5*(H_11 - H_00), and K_01 = (01|01).
    Return a dict with keys 'e_cc_corr', 't_amp', 'lambda_amp', 'P_cc_mo'.
    '''
    # TODO 13: Compute H_00, H_11, Delta_01, K_01 = eri_mo[0,1,0,1], solve the quadratic for t_amp,
    #          evaluate e_cc_corr = K_01 * t_amp, solve for lambda_amp = -K_01 / (2*Delta_01 + 2*K_01*t_amp),
    #          and form P_cc_mo.
    raise NotImplementedError("Complete TODO 13: solve_ccd_lambda_and_1rdm_2x2")


def bfgs_inverse_hessian_update(
    H_inv: np.ndarray, s_step: np.ndarray, y_grad_diff: np.ndarray
) -> np.ndarray:
    '''
    TODO 14: Perform one BFGS inverse-Hessian update on (n_dim, n_dim) matrix `H_inv`
    given step vector `s_step` and gradient difference `y_grad_diff`.
    If s_step @ y_grad_diff <= 1e-12, return H_inv unchanged.
    '''
    # TODO 14: Implement (I - rho * outer(s, y)) @ H_inv @ (I - rho * outer(y, s)) + rho * outer(s, s).
    raise NotImplementedError("Complete TODO 14: bfgs_inverse_hessian_update")


def run_geometry_optimization_1d(
    r_init_bohr: float,
    energy_and_grad_fn,
    max_iter: int = 25,
    grad_tol: float = 1e-6,
    max_step_bohr: float = 0.15,
) -> dict[str, float | int | list[float]]:
    '''
    TODO 15: Optimize a 1D bond length starting from `r_init_bohr` using quasi-Newton secant/BFGS steps
    and `energy_and_grad_fn(r) -> (energy, grad)`.
    Return a dict with keys 'converged', 'r_eq_bohr', 'e_min_eh', 'final_grad', 'n_steps', 'r_history', 'e_history'.
    '''
    # TODO 15: Iterate quasi-Newton steps (clipping step size to max_step_bohr) until |grad| < grad_tol.
    raise NotImplementedError("Complete TODO 15: run_geometry_optimization_1d")

## Part 5: Step-by-Step Checkpoint & Verification Suite

Run `run_section05d_checkpoints()` below as you implement each `# TODO` function to test your nuclear gradients, Pulay overlap forces, CPHF/Z-vector solver, analytical RHF and MP2 gradients against 5-point finite differences, CC $\Lambda$-equations, and BFGS geometry optimizer!

In [2]:
def run_section05d_checkpoints() -> None:
    '''Step-by-step checkpoint verifier for all 15 #TODO functions in Section 05d against PySCF gradients.'''
    passed = 0
    total_todos = 15

    try:
        charges_heh = np.array([2.0, 1.0])
        coords_heh = np.array([[0.0, 0.0, 0.0], [1.4632, 0.0, 0.0]])
        e_nuc_1, g_nuc_1 = compute_nuclear_repulsion_and_grad(charges_heh, coords_heh)
        assert np.isclose(e_nuc_1, 2.0 / 1.4632) and np.allclose(np.sum(g_nuc_1, axis=0), 0.0)
        assert np.isclose(g_nuc_1[1, 0], -2.0 / (1.4632**2))
        passed += 1
        print("[PASS] TODO 1 : compute_nuclear_repulsion_and_grad")

        s_00, ds_a, ds_b = gaussian_1d_overlap_and_center_deriv(0, 0, 0.0, 1.4, 0.5, 0.8)
        s_p, _, _ = gaussian_1d_overlap_and_center_deriv(0, 0, 1e-6, 1.4, 0.5, 0.8)
        s_m, _, _ = gaussian_1d_overlap_and_center_deriv(0, 0, -1e-6, 1.4, 0.5, 0.8)
        assert np.isclose(ds_a, (s_p - s_m) / 2e-6, atol=1e-8) and np.isclose(ds_a, -ds_b)
        passed += 1
        print("[PASS] TODO 2 : gaussian_1d_overlap_and_center_deriv")

        ints_d = compute_diatomic_sto3g_integrals_and_derivs(1.4632)
        assert ints_d["S"].shape == (2, 2) and ints_d["dS_dR"].shape == (2, 2) and ints_d["dS_dR"][0, 1] < 0.0
        passed += 1
        print("[PASS] TODO 3 : compute_diatomic_sto3g_integrals_and_derivs")

        mf_0 = scf.RHF(ints_d["mol"]).run(conv_tol=1e-12)
        D_0 = 0.5 * mf_0.make_rdm1()
        W_rhf = compute_rhf_energy_weighted_density_matrix(mf_0.mo_coeff, mf_0.mo_energy, n_occ=1)
        assert W_rhf.shape == (2, 2) and np.isclose(np.trace(W_rhf @ ints_d["S"]), 2.0 * mf_0.mo_energy[0], atol=1e-8)
        passed += 1
        print("[PASS] TODO 4 : compute_rhf_energy_weighted_density_matrix")

        g_rhf_dict = assemble_analytical_rhf_gradient_scalar(
            D_0, W_rhf, ints_d["dH_core_dR"], ints_d["dS_dR"], ints_d["deri_dR"], float(ints_d["de_nuc_dR"])
        )
        pyscf_rhf_grad = mf_0.nuc_grad_method().kernel()[1, 0]
        assert np.isclose(g_rhf_dict["grad_total"], pyscf_rhf_grad, atol=1e-6)
        passed += 1
        print(f"[PASS] TODO 5 : assemble_analytical_rhf_gradient_scalar (dE_RHF/dR = {g_rhf_dict['grad_total']:+.8f} matches PySCF {pyscf_rhf_grad:+.8f})")

        hf_comp = compare_hellmann_feynman_vs_full_analytical_grad(g_rhf_dict)
        assert hf_comp["pulay_ovlp_magnitude"] > 0.1
        passed += 1
        print(f"[PASS] TODO 6 : compare_hellmann_feynman_vs_full_analytical_grad (Pulay overlap term = {hf_comp['pulay_ovlp_magnitude']:.4f} Eh/Bohr)")

        def rhf_e_of_r(r_val: float) -> float:
            d = _heh_pyscf_integrals_at_r(r_val)
            return float(scf.RHF(d["mol"]).run(conv_tol=1e-12).e_tot)

        g_5pt, err_5pt = verify_gradient_5pt_stencil(rhf_e_of_r, 1.4632, g_rhf_dict["grad_total"], h_step=1e-3)
        assert err_5pt < 1e-7
        passed += 1
        print(f"[PASS] TODO 7 : verify_gradient_5pt_stencil (Analytical vs 5-pt FD error = {err_5pt:.2e} Eh/Bohr)")

        C_mo = mf_0.mo_coeff
        h_mo = C_mo.T @ ints_d["H_core"] @ C_mo
        eri_mo = np.einsum("uvls,up,vq,lr,st->pqrt", ints_d["eri"], C_mo, C_mo, C_mo, C_mo, optimize=True)
        mp2_d = compute_mp2_unrelaxed_densities(eri_mo, mf_0.mo_energy, n_occ=1)
        assert mp2_d["P_oo"].shape == (1, 1) and mp2_d["P_vv"].shape == (1, 1) and mp2_d["e_mp2_corr"] < 0.0
        passed += 1
        print("[PASS] TODO 8 : compute_mp2_unrelaxed_densities")

        A_cphf, L_ai = build_cphf_hessian_and_mp2_lagrangian(
            eri_mo, mf_0.mo_energy, mp2_d["t2"], mp2_d["tau_tilde"], mp2_d["P_oo"], mp2_d["P_vv"], n_occ=1
        )
        assert A_cphf.shape == (1, 1) and L_ai.shape == (1, 1) and A_cphf[0, 0] > 0.0
        passed += 1
        print("[PASS] TODO 9 : build_cphf_hessian_and_mp2_lagrangian")

        Z_ia, P_mp2_rel = solve_z_vector_and_relaxed_mp2_1rdm(A_cphf, L_ai, mp2_d["P_oo"], mp2_d["P_vv"])
        assert P_mp2_rel.shape == (2, 2) and np.allclose(P_mp2_rel, P_mp2_rel.T) and np.isclose(np.trace(P_mp2_rel), 0.0, atol=1e-12)
        passed += 1
        print(f"[PASS] TODO 10: solve_z_vector_and_relaxed_mp2_1rdm (Z_01 = {Z_ia[0,0]:+.6f})")

        W_mp2_mo = compute_mp2_energy_weighted_density_mo(eri_mo, mf_0.mo_energy, mp2_d["tau_tilde"], P_mp2_rel, n_occ=1)
        assert W_mp2_mo.shape == (2, 2)
        passed += 1
        print("[PASS] TODO 11: compute_mp2_energy_weighted_density_mo")

        mp2_g_res = assemble_analytical_mp2_gradient_scalar(1.4632)
        pyscf_mp2_grad = mp.MP2(mf_0).run().nuc_grad_method().kernel()[1, 0]
        assert np.isclose(mp2_g_res["grad_mp2_total"], pyscf_mp2_grad, atol=1e-5)
        passed += 1
        print(f"[PASS] TODO 12: assemble_analytical_mp2_gradient_scalar (dE_MP2/dR = {mp2_g_res['grad_mp2_total']:+.8f} matches PySCF {pyscf_mp2_grad:+.8f})")

        cc_lam = solve_ccd_lambda_and_1rdm_2x2(h_mo, eri_mo)
        e_fci_pyscf = fci.FCI(ints_d["mol"], mf_0.mo_coeff).kernel()[0]
        assert np.isclose(mf_0.e_tot + cc_lam["e_cc_corr"], e_fci_pyscf, atol=1e-7)
        assert np.isclose(np.trace(cc_lam["P_cc_mo"]), 2.0, atol=1e-12)
        passed += 1
        print(f"[PASS] TODO 13: solve_ccd_lambda_and_1rdm_2x2 (t = {cc_lam['t_amp']:+.6f}, lambda = {cc_lam['lambda_amp']:+.6f})")

        H_inv_1 = bfgs_inverse_hessian_update(np.eye(1), np.array([0.05]), np.array([0.10]))
        assert np.isclose(H_inv_1[0, 0], 0.5)
        passed += 1
        print("[PASS] TODO 14: bfgs_inverse_hessian_update")

        def rhf_eg(r_val: float) -> tuple[float, float]:
            d = compute_diatomic_sto3g_integrals_and_derivs(r_val)
            m_sol = scf.RHF(d["mol"]).run(conv_tol=1e-12)
            D_sol = 0.5 * m_sol.make_rdm1()
            W = compute_rhf_energy_weighted_density_matrix(m_sol.mo_coeff, m_sol.mo_energy, 1)
            gd = assemble_analytical_rhf_gradient_scalar(D_sol, W, d["dH_core_dR"], d["dS_dR"], d["deri_dR"], float(d["de_nuc_dR"]))
            return float(m_sol.e_tot), gd["grad_total"]

        opt_res = run_geometry_optimization_1d(1.60, rhf_eg, grad_tol=1e-6)
        assert opt_res["converged"] and abs(opt_res["final_grad"]) < 1e-6 and 1.40 < opt_res["r_eq_bohr"] < 1.50
        passed += 1
        print(f"[PASS] TODO 15: run_geometry_optimization_1d (Converged R_eq = {opt_res['r_eq_bohr']:.6f} Bohr in {opt_res['n_steps']} steps)")
        print(f"\nCONGRATULATIONS! All {passed}/{total_todos} Section 05d (Analytical Gradients) checkpoints passed!")

    except NotImplementedError as exc:
        print(f"\n[PROGRESS: {passed}/{total_todos} checkpoints passed] Next up -> {exc}")
    except AssertionError as exc:
        print(f"\n[ASSERTION FAILED after {passed}/{total_todos} checkpoints] -> {exc}")


# Uncomment and run as you implement each TODO:
# run_section05d_checkpoints()
print("Section 05d checkpoint suite loaded. Implement TODOs 1-15 and call `run_section05d_checkpoints()` to test against PySCF!")